# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides an interactive guide for loading and exploring the 'FAIR²' dataset using the `mlcroissant` library. The workflow covers metadata loading, record set access, data extraction, and exploratory data analysis (EDA), with a focus on referencing dataset entities by their `@id` fields as defined by the Croissant schema.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, not a dict
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, their `@id`, and corresponding fields. Each entity in Croissant is referenced by its unique `@id`.

In [ ]:
# List all available record sets and their fields by @id
record_sets = dataset.record_sets
if not record_sets:
    print('No record sets are defined in this dataset.')
else:
    print('Available record sets:')
    for record_set in record_sets:
        print(f"- @id: {record_set.id}")
        print(f"  Name: {record_set.name}")
        print(f"  Description: {getattr(record_set, 'description', 'No description')}")
        fields = getattr(record_set, 'fields', [])
        if fields:
            for field in fields:
                print(f"    Field @id: {field.id} | Name: {field.name} | Data Type: {getattr(field, 'data_type', '-')}")
        print()

### Inspect a Sample Record
Let's print out a single record for each available record set, referencing it by its `@id`.

In [ ]:
for record_set in record_sets:
    print(f"\nSample record from record set '@id': {record_set.id}")
    rec_iter = dataset.records(record_set=record_set.id)
    try:
        sample = next(rec_iter)
        print(json.dumps(sample, indent=2))
    except StopIteration:
        print("No records available.")

## 3. Data Extraction
Load all data from each available record set into pandas DataFrames for further analysis. Always use record set and field `@id` references used in the Croissant schema.

In [ ]:
# Extract data from all record sets into a dict of DataFrames
dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set.id] = df

# Show DataFrame columns for each record set
for record_set in dataframes:
    print(f"\nDataFrame for record set '@id': {record_set}")
    print("Columns:", dataframes[record_set].columns.tolist())
    display(dataframes[record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA tasks: filtering, normalization, grouping, outlier removal, etc. Example below assumes a numeric and grouping field exists. Update the `numeric_field_id` and `group_field_id` with correct `@id` references based on the overview.

If no record sets or numeric fields are available, this cell will demonstrate on a dummy DataFrame.

In [ ]:
# NOTE: Replace these with real `@id` values from printed record sets overview
# Example IDs (UPDATE as needed):
record_set_id = None
numeric_field_id = None
group_field_id = None

for rset in dataframes:
    df_columns = dataframes[rset].columns.tolist()
    # Try to pick the first numeric field (float or int columns)
    num_candidates = dataframes[rset].select_dtypes(include=['float', 'int']).columns.tolist()
    if num_candidates:
        record_set_id = rset
        numeric_field_id = num_candidates[0]
        # Try to pick a grouping/categorical field
        cat_candidates = dataframes[rset].select_dtypes(include=['object']).columns.tolist()
        group_field_id = cat_candidates[0] if cat_candidates else None
        break

if record_set_id is not None:
    print(f"Using record set: {record_set_id}")
    print(f"Numeric field: {numeric_field_id}")
    print(f"Group field: {group_field_id}")
    df = dataframes[record_set_id]

    # Remove outliers (just as an example, using 1st/99th percentiles)
    lower, upper = df[numeric_field_id].quantile([0.01, 0.99])
    filtered_df = df[(df[numeric_field_id] >= lower) & (df[numeric_field_id] <= upper)]
    print(f"Filtered records on {numeric_field_id} between {lower:.2f} and {upper:.2f} (excludes outliers):")
    print(filtered_df.head())

    # Normalize the numeric field
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by grouping field (if applicable)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field suitable for EDA found in available record sets.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to the group field, if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, inspect, and analyze a FAIR² dataset described by a Croissant schema using `mlcroissant`. Record sets and fields are referenced by their canonical `@id`.

- Data loading and metadata exploration can be performed seamlessly via the schema URL.
- Each record set, field, and column can be referenced and loaded dynamically using their `@id`.
- Common data analysis and transformations (filtering, normalization, grouping, and visualization) are straightforward in this workflow, once record set and field IDs are identified.

**Remember:** Always refer to Croissant schema documentation for field descriptions, data use limitations, and entity definitions.

_Notebook prepared as an interactive example for dataset processing with mlcroissant._